# U2 — one QTDB model for all BW/EM/MA noise subsets (512 samples)

This standalone Kaggle notebook trains **one** conditional DDPM model on a
balanced deterministic mix of seven nonempty artifact subsets: BW, EM, MA,
BW+EM, BW+MA, EM+MA, BW+EM+MA. It does **not** train three separate models.
Each noisy pair uses the same 512-sample QTDB clean segment, NSTDB artifact
patches and relative peak-to-peak level `r` across U0–U4. The three-source
patches are summed first and the resulting mixture is scaled once.

Architecture: HNF=none; timestep=film; attention=identity. The
conditional DDPM, multi-domain loss, 70/30 clean-segment train/validation
split, optimizer, EMA and 100-epoch budget match the 512-sample research
training setup. **Protocol 1** uses the first NSTDB half/channel 1 for
training; held-out QTDB records and the other noise half/channel belong to
the future common test evaluation. Test data do not select checkpoints.

Enable a Kaggle GPU and Internet. Optional input caches are discovered under
`/kaggle/input`; no source-code upload is required. If a session is stopped,
attach its prior output and rerun to resume from `last`.


## 1. Cài thư viện

In [ ]:
%pip -q install 'wfdb>=4.1,<5' 'scipy>=1.10' 'pandas>=2.0' 'matplotlib>=3.7' 'pyyaml>=6.0'


## 2. Reproducible configuration — select one artifact per Kaggle run


In [ ]:
import gc
import hashlib
import json
import math
import platform
import random
import shutil
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import wfdb


def wfdb_read_with_retry(operation, *args, **kwargs):
    """Retry transient PhysioNet errors without skipping or changing records."""
    for attempt in range(5):
        try:
            return operation(*args, **kwargs)
        except Exception as exc:
            message = str(exc).lower()
            transient = any(token in message for token in (
                '429', '500 error', '502', '503', '504', 'bad gateway',
                'timeout', 'timed out', 'connection error',
                'connection reset', 'temporarily unavailable',
            ))
            if not transient or attempt == 4:
                raise
            delay = min(2 ** (attempt + 1), 16)
            print(f'PhysioNet transient error ({exc}); retry {attempt + 1}/4 '
                  f'in {delay}s')
            time.sleep(delay)

SEED = 1234
ARTIFACTS_TO_TRAIN = ('MIX7',)  # One mixed model
PROTOCOLS_TO_TRAIN = (1,)
TARGET_EPOCHS = 100
# Safe default for one Kaggle version. Increase after measuring seconds/epoch.
EPOCHS_PER_EXPERIMENT_THIS_SESSION = 100

TEST_RECORDS = (
    'sel123', 'sel233', 'sel302', 'sel307', 'sel820', 'sel853',
    'sel16420', 'sel16795', 'sele0106', 'sele0121', 'sel32',
    'sel49', 'sel14046', 'sel15814',
)
TARGET_FS = 360
LENGTH = 512
MAX_UNPADDED = 496
INSERT_OFFSET = 16
PREPROCESSING_VERSION = 'qtdb_pu1_p_and_beat_40ms_reflect100ms_360hz_endpoints_pad16_v2'
MODEL_NORMALIZATION = 'per_segment_clean_maxabs_shared_with_noisy_v1'
LOSS_VERSION = 'l1_mean_relative_stft_snr_weighted_continuous_alpha_v1'
CHECKPOINT_TAG = 'mixed512_100ep_v1'
PHYSIONET_QT = 'qtdb/1.0.0'
PHYSIONET_NST = 'nstdb/1.0.0'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable a GPU accelerator in Kaggle before training.')

OUT = Path('/kaggle/working/qtdb_u2_all_noise_512_100ep')
TRAIN_DIR = OUT / 'checkpoints'
OUT.mkdir(parents=True, exist_ok=True)
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# None means the official full run. Use a small integer only for a smoke test.
TRAIN_RECORD_LIMIT = None
MAX_TRAIN_SEGMENTS = None

CONFIG = {
    'train': {'feats': 80, 'batch_size': 96, 'epochs': TARGET_EPOCHS, 'lr': 1e-4,
              'lambda_time': 1.0, 'lambda_freq': 0.1},
    'diffusion': {'beta_start': 1e-4, 'beta_end': 0.5,
                  'num_steps': 50, 'schedule': 'quad'},
    'model_normalization': MODEL_NORMALIZATION,
    'loss_version': LOSS_VERSION,
    'preprocessing_version': PREPROCESSING_VERSION,
    'data_protocol': 'qtdb_nstdb_balanced_7_nonempty_masks_512_v1',
    'architecture': {'id': 'U2', 'feature_block': 'two_conv3_relu',
                     'timestep': 'sinusoidal_film_scale_shift', 'attention': 'identity'},
}

VALID_ARTIFACTS = {'BW': 'bw', 'EM': 'em', 'MA': 'ma'}
if ARTIFACTS_TO_TRAIN != ('MIX7',):
    raise ValueError("This notebook requires ARTIFACTS_TO_TRAIN=('MIX7',)")
if not PROTOCOLS_TO_TRAIN or not set(PROTOCOLS_TO_TRAIN).issubset({1, 2}):
    raise ValueError('PROTOCOLS_TO_TRAIN must contain protocol 1 and/or 2')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
(OUT / 'config.json').write_text(json.dumps({
    'artifacts_to_train': ARTIFACTS_TO_TRAIN,
    'protocols_to_train': PROTOCOLS_TO_TRAIN,
    'epochs_per_experiment_this_session': EPOCHS_PER_EXPERIMENT_THIS_SESSION,
    **CONFIG,
}, indent=2), encoding='utf-8')
print('GPU:', torch.cuda.get_device_name(0), '| output:', OUT)


## 3. U2 architecture: plain Conv1D, original FiLM, no attention


In [ ]:
import torch.nn as nn


class ConvBlock1D(nn.Module):
    """Two ordinary same-length Conv1D layers; no HNF or half normalization."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class BridgeBlock(nn.Module):
    """FiLM‑based conditioning on noise level (sqrt(alpha_bar))."""
    def __init__(self, features, emb_dim=128):
        super().__init__()
        self.emb_dim = emb_dim
        self.film = nn.Sequential(
            nn.Linear(emb_dim, features * 2),
            nn.SiLU()
        )

    def forward(self, x, alpha_bar):
        # alpha_bar: (B,) or scalar
        # Ép phẳng alpha_bar về dạng 1D (Batch,) để tránh bị dư chiều -- CỰC QUAN TRỌNG
        alpha_bar = alpha_bar.view(-1)
        emb = self.sinusoidal_embedding(alpha_bar)
        scale_shift = self.film(emb)   # (B, 2*features)
        scale, shift = scale_shift.chunk(2, dim=1)
        scale = scale.unsqueeze(-1)
        shift = shift.unsqueeze(-1)
        return x * (1 + scale) + shift

    def sinusoidal_embedding(self, x):
        if x.dim() == 0:
            x = x.unsqueeze(0)
        device = x.device
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x.unsqueeze(-1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb


class UNet1D(nn.Module):
    """U2: same four-level conditional diffusion U-Net scaffold as U4."""

    def __init__(self, in_channels=2, base_channels=80, emb_dim=128,
                 out_channels=1):
        super().__init__()
        c = base_channels
        self.enc1 = ConvBlock1D(in_channels, c)
        self.bridge1 = BridgeBlock(c, emb_dim)
        self.down1 = nn.Conv1d(c, c * 2, 4, stride=2, padding=1)
        self.enc2 = ConvBlock1D(c * 2, c * 2)
        self.bridge2 = BridgeBlock(c * 2, emb_dim)
        self.down2 = nn.Conv1d(c * 2, c * 4, 4, stride=2, padding=1)
        self.enc3 = ConvBlock1D(c * 4, c * 4)
        self.bridge3 = BridgeBlock(c * 4, emb_dim)
        self.down3 = nn.Conv1d(c * 4, c * 8, 4, stride=2, padding=1)
        self.enc4 = ConvBlock1D(c * 8, c * 8)
        self.bridge4 = BridgeBlock(c * 8, emb_dim)
        self.attn = nn.Identity()
        self.up4 = nn.ConvTranspose1d(c * 8, c * 4, 4, stride=2, padding=1)
        self.dec4 = ConvBlock1D(c * 8, c * 4)
        self.up3 = nn.ConvTranspose1d(c * 4, c * 2, 4, stride=2, padding=1)
        self.dec3 = ConvBlock1D(c * 4, c * 2)
        self.up2 = nn.ConvTranspose1d(c * 2, c, 4, stride=2, padding=1)
        self.dec2 = ConvBlock1D(c * 2, c)
        self.final = nn.Conv1d(c, out_channels, 1)

    def forward(self, x, cond, noise_scale):
        if x.shape != cond.shape or x.shape[-1] % 8:
            raise ValueError('Expected matching [B,1,L] inputs with L divisible by 8')
        e1 = self.bridge1(self.enc1(torch.cat([x, cond], dim=1)), noise_scale)
        e2 = self.bridge2(self.enc2(self.down1(e1)), noise_scale)
        e3 = self.bridge3(self.enc3(self.down2(e2)), noise_scale)
        e4 = self.attn(self.bridge4(self.enc4(self.down3(e3)), noise_scale))
        d4 = self.dec4(torch.cat([self.up4(e4), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        return self.final(d2)


with torch.no_grad():
    probe_model = UNet1D(2, CONFIG['train']['feats'], 128, 1).to(DEVICE)
    probe_input = torch.zeros(2, 1, LENGTH, device=DEVICE)
    probe_output = probe_model(probe_input, probe_input,
                               torch.ones(2, 1, device=DEVICE))
    assert probe_output.shape == probe_input.shape
    print('U2 preflight:', tuple(probe_output.shape), '| parameters:',
          sum(p.numel() for p in probe_model.parameters()))
    del probe_model, probe_input, probe_output


## 4. Conditional DDPM và loss đa miền

In [ ]:
from functools import partial
from inspect import isfunction
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial
from inspect import isfunction

# ----- Bắt buộc phải có torchaudio hoặc dùng torch.stft -----
# Nếu không có torchaudio, ta dùng torch.stft nguyên bản
def stft_loss(pred, target, n_fft=128, hop_length=64):
    """
    Tính MSE loss trên phổ STFT giữa pred và target.
    """
    # (B, C, L) -> (B*C, L)
    B, C, L = pred.shape
    pred = pred.view(-1, L)
    target = target.view(-1, L)
    
    # STFT: returns (B, freq_bins, time_frames, 2)
    spec_pred = torch.stft(pred, n_fft=n_fft, hop_length=hop_length, 
                           return_complex=True)
    spec_target = torch.stft(target, n_fft=n_fft, hop_length=hop_length,
                             return_complex=True)
    
    # Lấy magnitude (độ lớn)
    mag_pred = torch.abs(spec_pred)
    mag_target = torch.abs(spec_target)
    
    return F.mse_loss(mag_pred, mag_target)


def exists(x):
    return x is not None

def default(val, d):
    if exists(val):
        return val
    return d() if isfunction(d) else d


class DDPM(nn.Module):
    def __init__(self, base_model, config, device, conditional=True):
        super().__init__()
        self.device = device
        self.model = base_model
        self.config = config
        self.conditional = conditional
        
        # ----------------------
        # 👉 THÊM: Loss weights
        # ----------------------
        self.lambda_time = config['train'].get('lambda_time', 1.0)
        self.lambda_freq = config['train'].get('lambda_freq', 0.1)  # cân bằng
        
        self.loss_func_l1 = nn.L1Loss(reduction='sum').to(device)
        
        config_diff = config["diffusion"]
        self.num_steps = config_diff["num_steps"]
        self.set_new_noise_schedule(config_diff, device)
        
    def make_beta_schedule(self, schedule='linear', n_timesteps=1000, start=1e-5, end=1e-2):
        if schedule == 'linear':
            betas = torch.linspace(start, end, n_timesteps)
        elif schedule == "quad":
            betas = torch.linspace(start ** 0.5, end ** 0.5, n_timesteps) ** 2
        elif schedule == "sigmoid":
            betas = torch.linspace(-6, 6, n_timesteps)
            betas = torch.sigmoid(betas) * (end - start) + start
        return betas
    
    def set_new_noise_schedule(self, config_diff, device):
        to_torch = partial(torch.tensor, dtype=torch.float32, device=device)
        
        betas = self.make_beta_schedule(
            schedule=config_diff["schedule"], 
            n_timesteps=config_diff["num_steps"],
            start=config_diff["beta_start"], 
            end=config_diff["beta_end"]
        )
        betas = betas.detach().cpu().numpy() if isinstance(betas, torch.Tensor) else betas
        
        alphas = 1. - betas
        alphas_cumprod = np.cumprod(alphas, axis=0)
        alphas_cumprod_prev = np.append(1., alphas_cumprod[:-1])
        self.sqrt_alphas_cumprod_prev = np.sqrt(np.append(1., alphas_cumprod))
        
        self.register_buffer('betas', to_torch(betas))
        self.register_buffer('alphas_cumprod', to_torch(alphas_cumprod))
        self.register_buffer('alphas_cumprod_prev', to_torch(alphas_cumprod_prev))
        self.register_buffer('sqrt_alphas_cumprod', to_torch(np.sqrt(alphas_cumprod)))
        self.register_buffer('sqrt_one_minus_alphas_cumprod', to_torch(np.sqrt(1. - alphas_cumprod)))
        self.register_buffer('log_one_minus_alphas_cumprod', to_torch(np.log(1. - alphas_cumprod)))
        self.register_buffer('sqrt_recip_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod)))
        self.register_buffer('sqrt_recipm1_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod - 1)))
        
        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)
        self.register_buffer('posterior_variance', to_torch(posterior_variance))
        self.register_buffer('posterior_log_variance_clipped', 
                             to_torch(np.log(np.maximum(posterior_variance, 1e-20))))
        self.register_buffer('posterior_mean_coef1', 
                             to_torch(betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod)))
        self.register_buffer('posterior_mean_coef2', 
                             to_torch((1. - alphas_cumprod_prev) * np.sqrt(alphas) / (1. - alphas_cumprod)))
        
    def predict_start_from_noise(self, x_t, t, noise):
        # Nắn lại chiều tensor hệ số thành (Batch, 1, 1) để nhân đúng với tensor tín hiệu 3D
        coef1 = self.sqrt_recip_alphas_cumprod[t].view(-1, 1, 1)
        coef2 = self.sqrt_recipm1_alphas_cumprod[t].view(-1, 1, 1)
        
        return coef1 * x_t - coef2 * noise
    
    def q_posterior(self, x_start, x_t, t):
        posterior_mean = self.posterior_mean_coef1[t] * x_start + \
                         self.posterior_mean_coef2[t] * x_t
        posterior_log_variance_clipped = self.posterior_log_variance_clipped[t]
        return posterior_mean, posterior_log_variance_clipped
    
    def p_mean_variance(self, x, t, clip_denoised: bool, condition_x=None):
        batch_size = x.shape[0]
        noise_level = torch.FloatTensor(
            [self.sqrt_alphas_cumprod_prev[t+1]]
        ).repeat(batch_size, 1).to(x.device)
        if condition_x is not None:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.model(x, condition_x, noise_level))
        else:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.model(x, noise_level))
        if clip_denoised:
            x_recon.clamp_(-1., 1.)
        model_mean, posterior_log_variance = self.q_posterior(
            x_start=x_recon, x_t=x, t=t)
        return model_mean, posterior_log_variance
    
    @torch.no_grad()
    def p_sample(self, x, t, clip_denoised=False, condition_x=None):
        model_mean, model_log_variance = self.p_mean_variance(
            x=x, t=t, clip_denoised=clip_denoised, condition_x=condition_x)
        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        return model_mean + noise * (0.5 * model_log_variance).exp()

    @torch.no_grad()
    def p_sample_loop(self, x_in, continous=False):
        device = self.betas.device
        sample_inter = (1 | (self.num_steps//10))
        if not self.conditional:
            shape = x_in
            cur_x = torch.randn(shape, device=device)
            ret_x = cur_x
            for i in reversed(range(0, self.num_steps)):
                cur_x = self.p_sample(cur_x, i)
                if i % sample_inter == 0:
                    ret_x = torch.cat([ret_x, cur_x], dim=0)
        else:
            x = x_in
            shape = x.shape
            cur_x = torch.randn(shape, device=device)
            ret_x = [cur_x]
            for i in reversed(range(0, self.num_steps)):
                cur_x = self.p_sample(cur_x, i, condition_x=x)
                if i % sample_inter == 0:
                    ret_x.append(cur_x)
        if continous:
            return ret_x
        else:
            return ret_x[-1]    
    
    @torch.no_grad()
    def sample(self, batch_size=1, shape=[1, 512], continous=False):
        return self.p_sample_loop((batch_size, shape[0], shape[1]), continous)
    
    # ==========================================================
    # 👉 DDIM SAMPLER (đã có sẵn trong source gốc)
    # ==========================================================
    @torch.no_grad()
    def ddim_sample_loop(self, x_in, ddim_timesteps=50, ddim_eta=0.0, num_shots=1):
        device = self.betas.device
        T = self.num_steps
        S = ddim_timesteps
        tau = [int(np.floor((T / (S ** 2)) * (i ** 2))) for i in range(S + 1)]
        
        alphas_cumprod_with_1 = torch.cat([
            torch.tensor([1.0], dtype=self.alphas_cumprod.dtype, device=device),
            self.alphas_cumprod
        ]).float()
        
        out_accum = 0.0
        for shot in range(num_shots):
            shape = x_in.shape
            x = torch.randn(shape, device=device)
            for i in reversed(range(1, S + 1)):
                t = tau[i]
                t_prev = tau[i - 1]
                noise_level = torch.FloatTensor(
                    [self.sqrt_alphas_cumprod_prev[t]]
                ).repeat(x.shape[0], 1).to(device)
                if not self.conditional:
                    eps = self.model(x, noise_level)
                else:
                    eps = self.model(x, x_in, noise_level)
                    
                a_t = alphas_cumprod_with_1[t]
                a_t_prev = alphas_cumprod_with_1[t_prev]
                x0_pred = (x - torch.sqrt(1. - a_t) * eps) / torch.sqrt(a_t)
                if t_prev == 0:
                    sigma_t = 0.0
                else:
                    sigma_t = ddim_eta * torch.sqrt((1. - a_t_prev) / (1. - a_t)) * \
                              torch.sqrt(torch.clamp(1. - a_t / a_t_prev, min=0.0))
                direction = torch.sqrt(torch.clamp(1. - a_t_prev - sigma_t**2, min=0.0)) * eps
                noise = sigma_t * torch.randn_like(x) if (ddim_eta > 0 and t_prev > 0) else 0.0
                x = torch.sqrt(a_t_prev) * x0_pred + direction + noise
            out_accum += x
        return out_accum / num_shots

    @torch.no_grad()
    def denoising(self, x_in, continous=False, use_ddim=False, ddim_steps=50, ddim_eta=0.0, num_shots=1):
        if use_ddim:
            return self.ddim_sample_loop(x_in, ddim_timesteps=ddim_steps, ddim_eta=ddim_eta, num_shots=num_shots)
        else:
            if num_shots > 1:
                out_accum = 0.0
                for _ in range(num_shots):
                    out_accum += self.p_sample_loop(x_in, continous)
                return out_accum / num_shots
            return self.p_sample_loop(x_in, continous)
    
    def q_sample_loop(self, x_start, continous=False):
        sample_inter = (1 | (self.num_steps//10))
        ret_x = [x_start]
        cur_x = x_start
        for t in range(1, self.num_steps+1):
            B,C,L = cur_x.shape
            continuous_sqrt_alpha_cumprod = torch.FloatTensor(
                np.random.uniform(
                    self.sqrt_alphas_cumprod_prev[t-1],
                    self.sqrt_alphas_cumprod_prev[t],
                    size=B
                )
            ).to(cur_x.device)
            continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(B, -1)
            noise = torch.randn_like(cur_x)
            cur_x = self.q_sample(
                x_start=cur_x, 
                continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1), 
                noise=noise
            )
            if t % sample_inter == 0:
                ret_x.append(cur_x)
        if continous:
            return ret_x
        else:
            return ret_x[-1]
    
    def q_sample(self, x_start, continuous_sqrt_alpha_cumprod, noise=None):
        noise = default(noise, lambda: torch.randn_like(x_start))
        return continuous_sqrt_alpha_cumprod * x_start + \
               (1 - continuous_sqrt_alpha_cumprod**2).sqrt() * noise
    
    # ==========================================================
    # 👉 HÀM LOSS CHÍNH (ĐÃ THÊM STFT)
    # ==========================================================
    def p_losses(self, x_in, y_in, noise=None):
        """
        Multi‑domain Loss:
        - L1 trên noise (miền thời gian)
        - STFT Loss trên tín hiệu dự đoán (miền tần số)
        """
        x_start = x_in
        B, C, L = x_start.shape

        # Lấy ngẫu nhiên các giá trị t khác nhau cho từng mẫu trong Batch, trực tiếp trên thiết bị (GPU)
        t = torch.randint(0, self.num_steps, (B,), device=x_start.device).long()
        # 1. Chuyển mảng NumPy sang PyTorch Tensor và đẩy lên cùng GPU với t
        sqrt_alphas_prev_tensor = torch.tensor(self.sqrt_alphas_cumprod_prev, dtype=torch.float32, device=t.device)
        # 2. Lấy giới hạn dưới và trên. 
        # (Lưu ý: Do t giờ chạy từ 0 đến num_steps-1, ta dùng chỉ số t và t+1 để thay thế cho t-1 và t nhằm tránh lỗi âm)
        lower = sqrt_alphas_prev_tensor[t]
        upper = sqrt_alphas_prev_tensor[t+1]
        # 3. Sinh số ngẫu nhiên uniform hoàn toàn bằng PyTorch trực tiếp trên GPU
        continuous_sqrt_alpha_cumprod = lower + torch.rand(B, device=t.device) * (upper - lower)
        continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(B, -1)
        noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(
            x_start=x_start, 
            continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1), 
            noise=noise
        )

        # Dự đoán noise
        if not self.conditional:
            x_recon = self.model(x_noisy, continuous_sqrt_alpha_cumprod)
        else:
            x_recon = self.model(x_noisy, y_in, continuous_sqrt_alpha_cumprod)

        # ---------- 1. L1 Loss (miền thời gian) ----------
        loss_l1 = self.loss_func_l1(noise, x_recon)  # (B*C*L)

        # ---------- 2. STFT Loss (miền tần số) ----------
        # Dự đoán x0 từ noise đã dự đoán
        x0_pred = self.predict_start_from_noise(x_noisy, t, x_recon)
        
        # Tính STFT loss giữa x0_pred và x_start (clean gốc)
        loss_stft = stft_loss(x0_pred, x_start, n_fft=128, hop_length=64)
        
        # ---------- 3. Tổng hợp ----------
        total_loss = self.lambda_time * loss_l1 + self.lambda_freq * loss_stft
        return total_loss
    
    def forward(self, x, y, *args, **kwargs):
        return self.p_losses(x, y, *args, **kwargs)


class EMA(object):
    def __init__(self, mu=0.999):
        self.mu = mu
        self.shadow = {}

    def register(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name].data = (1. - self.mu) * param.data + self.mu * self.shadow[name].data

    def ema(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.shadow[name].data)

    def ema_copy(self, module):
        module_copy = type(module)(module.config).to(module.config.device)
        module_copy.load_state_dict(module.state_dict())
        self.ema(module_copy)
        return module_copy

    def state_dict(self):
        return self.shadow

    def load_state_dict(self, state_dict):
        self.shadow = state_dict

class ResearchLossDDPM(DDPM):
    """Same DDPM/UNet architecture, with a scale-stable multi-domain objective."""

    def loss_components(self, x_in, y_in, noise=None):
        x_start = x_in
        batch = x_start.shape[0]
        timestep = torch.randint(0, self.num_steps, (batch,), device=x_start.device)
        boundaries = torch.as_tensor(
            self.sqrt_alphas_cumprod_prev, dtype=x_start.dtype, device=x_start.device
        )
        lower, upper = boundaries[timestep], boundaries[timestep + 1]
        continuous = lower + torch.rand(batch, device=x_start.device) * (upper - lower)
        continuous_2d = continuous.view(batch, 1)
        continuous_3d = continuous.view(batch, 1, 1)
        gaussian_noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(x_start, continuous_3d, gaussian_noise)
        predicted_noise = self.model(x_noisy, y_in, continuous_2d)

        # Mean keeps the time loss independent of batch size and 512-sample length.
        loss_time = F.l1_loss(predicted_noise, gaussian_noise, reduction='mean')

        # Reconstruct x0 with the exact continuous alpha used to create x_t.
        residual_scale = torch.sqrt(torch.clamp(1.0 - continuous_3d ** 2, min=0.0))
        x0_prediction = (x_noisy - residual_scale * predicted_noise) / continuous_3d.clamp_min(1e-6)

        n_fft, hop_length = 128, 64
        window = torch.hann_window(n_fft, dtype=x_start.dtype, device=x_start.device)
        pred_flat = x0_prediction.reshape(-1, x0_prediction.shape[-1])
        target_flat = x_start.reshape(-1, x_start.shape[-1])
        pred_mag = torch.abs(torch.stft(pred_flat, n_fft=n_fft, hop_length=hop_length,
                                         window=window, return_complex=True))
        target_mag = torch.abs(torch.stft(target_flat, n_fft=n_fft, hop_length=hop_length,
                                           window=window, return_complex=True))
        pred_mag = pred_mag.reshape(batch, -1)
        target_mag = target_mag.reshape(batch, -1)
        spectral_error = torch.mean((pred_mag - target_mag) ** 2, dim=1)
        target_power = torch.mean(target_mag ** 2, dim=1).clamp_min(1e-6)
        relative_frequency = spectral_error / target_power

        # x0 is ill-conditioned at high-noise timesteps, so weight STFT by diffusion SNR.
        alpha_bar = continuous ** 2
        diffusion_snr = alpha_bar / torch.clamp(1.0 - alpha_bar, min=1e-6)
        frequency_weight = torch.clamp(diffusion_snr, max=1.0)
        loss_frequency = torch.mean(relative_frequency * frequency_weight)
        total = self.lambda_time * loss_time + self.lambda_freq * loss_frequency
        return {'total': total, 'time': loss_time, 'frequency': loss_frequency,
                'frequency_unweighted': relative_frequency.mean(),
                'frequency_weight': frequency_weight.mean()}

    def p_losses(self, x_in, y_in, noise=None):
        return self.loss_components(x_in, y_in, noise)['total']


## 5. Hàm tạo QTDB clean segments

In [ ]:
BEAT_SYMBOLS = set('NLRBAaJSVrFejnE/fQ?')

def p_onsets(annotation):
    symbols = np.asarray(annotation.symbol)
    samples = np.asarray(annotation.sample, dtype=np.int64)
    return np.asarray([int(samples[i]) for i, symbol in enumerate(symbols)
                       if symbol == '(' and 'p' in symbols[i + 1:i + 4]], dtype=np.int64)

def r_locations(annotation):
    return np.asarray([int(sample) for sample, symbol in zip(annotation.sample, annotation.symbol)
                       if symbol in BEAT_SYMBOLS], dtype=np.int64)

def resample_reflect(segment, original_fs):
    pad = min(max(1, int(round(0.1 * original_fs))), len(segment) - 1)
    padded = np.pad(segment, (pad, pad), mode='reflect')
    new_length = int(round(len(padded) * TARGET_FS / original_fs))
    resampled = signal.resample(padded, new_length).astype(np.float32)
    target_pad = int(round(pad * TARGET_FS / original_fs))
    expected = int(round(len(segment) * TARGET_FS / original_fs))
    return resampled[target_pad:target_pad + expected]

def clean_segments_for_record(name):
    record = wfdb_read_with_retry(wfdb.rdrecord, name, pn_dir=PHYSIONET_QT, channels=[0])
    # QTDB does not provide .atr for every record (e.g. sel30).
    # .pu1 provides P boundaries and beat fiducials throughout each record.
    pu = wfdb_read_with_retry(wfdb.rdann, name, 'pu1', pn_dir=PHYSIONET_QT)
    fs = float(record.fs)
    ecg = record.p_signal[:, 0].astype(np.float32)
    starts = p_onsets(pu) - int(round(0.04 * fs))
    starts = starts[(starts >= 0) & (starts < len(ecg))]
    peaks = r_locations(pu)
    beats, rows = [], []
    audit = {'record_id': name, 'p_onsets': len(starts),
             'candidate_intervals': max(0, len(starts) - 1),
             'rejected_short_interval': 0, 'rejected_multiple_r': 0,
             'rejected_length': 0, 'rejected_nonfinite_or_flat': 0}
    resampled_lengths = []
    for segment_index, (start, end) in enumerate(zip(starts[:-1], starts[1:])):
        if end <= start + 2:
            audit['rejected_short_interval'] += 1
            continue
        r_count = int(np.count_nonzero((peaks >= start) & (peaks < end)))
        if r_count > 1:
            audit['rejected_multiple_r'] += 1
            continue
        segment = resample_reflect(ecg[start:end], fs)
        resampled_lengths.append(len(segment))
        if not 2 <= len(segment) <= MAX_UNPADDED:
            audit['rejected_length'] += 1
            continue
        segment = segment - 0.5 * (segment[0] + segment[-1])
        clean = np.zeros(LENGTH, dtype=np.float32)
        clean[INSERT_OFFSET:INSERT_OFFSET + len(segment)] = segment
        if not np.all(np.isfinite(clean)) or np.ptp(clean) <= 0:
            audit['rejected_nonfinite_or_flat'] += 1
            continue
        beats.append(clean)
        rows.append({'sample_id': f'{name}:{segment_index}', 'patient_id': name,
                     'record_id': name, 'clean_segment_id': f'{name}:{segment_index}',
                     'start_sample': int(start), 'end_sample': int(end),
                     'unpadded_length': len(segment), 'r_count': r_count,
                     'annotation_source': 'pu1'})
    audit['min_resampled_length'] = min(resampled_lengths) if resampled_lengths else np.nan
    audit['max_resampled_length'] = max(resampled_lengths) if resampled_lengths else np.nan
    audit['accepted_segments'] = len(beats)
    return beats, rows, audit



## 6. Cache helpers

In [ ]:
def cached_input(filename):
    local = OUT / filename
    if local.is_file():
        return local
    root = Path('/kaggle/input')
    matches = sorted(root.rglob(filename)) if root.exists() else []
    if len(matches) > 1:
        raise RuntimeError(f'Multiple files named {filename}; attach only one prior output')
    if matches:
        shutil.copy2(matches[0], local)
        return local
    return None


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


## 7. Tạo QTDB train/validation

In [ ]:
clean_cache = cached_input('train_clean_full.npy') if TRAIN_RECORD_LIMIT is None else None
meta_cache = cached_input('train_metadata_full.csv') if TRAIN_RECORD_LIMIT is None else None
audit_cache = cached_input('train_record_audit.csv') if TRAIN_RECORD_LIMIT is None else None
if any(path is not None for path in (clean_cache, meta_cache, audit_cache)) and not all(
        path is not None for path in (clean_cache, meta_cache, audit_cache)):
    raise RuntimeError('Incomplete QTDB cache: clean, metadata and audit must travel together')

if clean_cache is not None:
    TRAIN_CLEAN = np.load(clean_cache).astype(np.float32, copy=False)
    TRAIN_META = pd.read_csv(meta_cache)
    TRAIN_AUDIT = pd.read_csv(audit_cache)
    TRAIN_RECORDS = tuple(TRAIN_AUDIT.record_id)
else:
    records = wfdb_read_with_retry(wfdb.get_record_list, 'qtdb')
    TRAIN_RECORDS = tuple(sorted(set(records) - set(TEST_RECORDS)))
    if TRAIN_RECORD_LIMIT is not None:
        TRAIN_RECORDS = TRAIN_RECORDS[:TRAIN_RECORD_LIMIT]
    beats, rows, audits = [], [], []
    for record_name in tqdm(TRAIN_RECORDS, desc='QTDB train records'):
        record_beats, record_rows, audit = clean_segments_for_record(record_name)
        beats.extend(record_beats)
        rows.extend(record_rows)
        audits.append(audit)
    if not beats:
        raise RuntimeError('No QTDB training segments were produced')
    TRAIN_CLEAN = np.stack(beats).astype(np.float32)
    TRAIN_META = pd.DataFrame(rows)
    TRAIN_AUDIT = pd.DataFrame(audits)
    if MAX_TRAIN_SEGMENTS is not None:
        TRAIN_CLEAN = TRAIN_CLEAN[:MAX_TRAIN_SEGMENTS]
        TRAIN_META = TRAIN_META.iloc[:MAX_TRAIN_SEGMENTS].reset_index(drop=True)
    if TRAIN_RECORD_LIMIT is None and MAX_TRAIN_SEGMENTS is None:
        np.save(OUT / 'train_clean_full.npy', TRAIN_CLEAN)
        TRAIN_META.to_csv(OUT / 'train_metadata_full.csv', index=False)
        TRAIN_AUDIT.to_csv(OUT / 'train_record_audit.csv', index=False)

assert TRAIN_CLEAN.ndim == 2 and TRAIN_CLEAN.shape[1] == LENGTH
assert len(TRAIN_CLEAN) == len(TRAIN_META) and TRAIN_META.sample_id.is_unique
assert np.isfinite(TRAIN_CLEAN).all()
assert not set(TRAIN_META.record_id) & set(TEST_RECORDS)

# Preserve the existing benchmark contract: deterministic 70/30 segment split.
indices = np.random.RandomState(SEED).permutation(len(TRAIN_CLEAN))
cut = int(0.70 * len(indices))
TRAIN_INDICES, VAL_INDICES = indices[:cut], indices[cut:]
split = np.where(np.isin(np.arange(len(TRAIN_CLEAN)), TRAIN_INDICES), 'train', 'validation')
split_frame = TRAIN_META[['sample_id', 'record_id']].copy()
split_frame['split'] = split
split_frame.to_csv(OUT / 'train_validation_split.csv', index=False)
print('QTDB records:', len(TRAIN_RECORDS), '| segments:', len(TRAIN_CLEAN),
      '| train:', len(TRAIN_INDICES), '| validation:', len(VAL_INDICES))


## 8. Tải ba artifact NSTDB

In [ ]:
NOISE = {}
NOISE_MIDPOINT = {}
for artifact, record_name in VALID_ARTIFACTS.items():
    filename = f'{record_name}_360.npy'
    cache = cached_input(filename)
    if cache is not None:
        values = np.load(cache).astype(np.float32, copy=False)
    else:
        record = wfdb_read_with_retry(wfdb.rdrecord, record_name, pn_dir=PHYSIONET_NST)
        values = record.p_signal.astype(np.float32)
        if int(record.fs) != TARGET_FS:
            new_length = int(round(len(values) * TARGET_FS / float(record.fs)))
            values = signal.resample(values, new_length, axis=0).astype(np.float32)
        np.save(OUT / filename, values)
    if values.ndim != 2 or values.shape[1] < 2 or not np.isfinite(values).all():
        raise RuntimeError(f'Invalid NSTDB {artifact} record: {values.shape}')
    NOISE[artifact] = values
    NOISE_MIDPOINT[artifact] = len(values) // 2
    print(artifact, values.shape, 'midpoint=', NOISE_MIDPOINT[artifact])


## 9. Fixed seven-subset mixed-noise dataset and fingerprint


In [ ]:
# One immutable seven-subset mixture assignment for every Ui.
# The seven masks are BW, EM, MA, BW+EM, BW+MA, EM+MA, BW+EM+MA.
MIX_COMPONENTS = ('BW', 'EM', 'MA')
MIX_MASKS = np.asarray([
    (1, 0, 0), (0, 1, 0), (0, 0, 1),
    (1, 1, 0), (1, 0, 1), (0, 1, 1), (1, 1, 1),
], dtype=np.uint8)
MIX_NAMES = ('BW', 'EM', 'MA', 'BW+EM', 'BW+MA', 'EM+MA', 'BW+EM+MA')
mask_pool = np.tile(np.arange(7, dtype=np.uint8),
                    (len(TRAIN_CLEAN) + 6) // 7)[:len(TRAIN_CLEAN)]
MIX_MASK_IDS = np.random.RandomState(SEED + 404).permutation(mask_pool)
MIX_LEVELS = (np.random.RandomState(SEED).randint(
    20, 200, size=len(TRAIN_CLEAN)).astype(np.float32) / 100.0)
assert np.all(MIX_MASKS.sum(axis=1) >= 1)
assert len(MIX_MASK_IDS) == len(MIX_LEVELS) == len(TRAIN_CLEAN)
for split_name, split_indices in (('train', TRAIN_INDICES),
                                  ('validation', VAL_INDICES)):
    counts = np.bincount(MIX_MASK_IDS[split_indices], minlength=7)
    assert np.all(counts > 0), (split_name, counts)
    print(split_name, dict(zip(MIX_NAMES, counts.tolist())))


class FixedMixedDataset(Dataset):
    """Deterministic clean/noisy pairs; all Ui see the same seven mask classes."""

    def __init__(self, clean_array, indices, protocol):
        self.clean = clean_array
        self.indices = np.asarray(indices, dtype=np.int64)
        self.protocol = int(protocol)
        train_channel = 0 if protocol == 1 else 1
        self.noise = {}
        self.patch_count = {}
        for artifact in MIX_COMPONENTS:
            values = NOISE[artifact][:NOISE_MIDPOINT[artifact], train_channel]
            self.noise[artifact] = values
            self.patch_count[artifact] = len(values) // LENGTH
            if self.patch_count[artifact] == 0:
                raise RuntimeError(f'{artifact}: noise training half too short')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        source_index = int(self.indices[item])
        clean_raw = self.clean[source_index]
        mask = MIX_MASKS[int(MIX_MASK_IDS[source_index])]
        patches = []
        for enabled, artifact in zip(mask, MIX_COMPONENTS):
            if enabled:
                start = (source_index % self.patch_count[artifact]) * LENGTH
                patches.append(self.noise[artifact][start:start + LENGTH])
        mixed_patch = np.sum(np.stack(patches, axis=0), axis=0)
        clean_range = float(np.ptp(clean_raw))
        noise_range = float(np.ptp(mixed_patch))
        if clean_range <= 0 or noise_range <= 0:
            raise ValueError(f'Zero range at QTDB sample {source_index}')
        noisy_raw = (clean_raw + float(MIX_LEVELS[source_index])
                     * clean_range / noise_range * mixed_patch)
        scale = max(float(np.max(np.abs(clean_raw))), 1e-8)
        clean = torch.from_numpy((clean_raw / scale).astype(np.float32))[None, :]
        noisy = torch.from_numpy((noisy_raw / scale).astype(np.float32))[None, :]
        return clean, noisy


def make_loader(artifact, protocol, indices, shuffle, drop_last, epoch=0):
    if artifact != 'MIX7':
        raise ValueError('This notebook trains one MIX7 model, not three specialists')
    dataset = FixedMixedDataset(TRAIN_CLEAN, indices, protocol)
    generator = torch.Generator().manual_seed(
        SEED + 44 * 100_000 + protocol * 1_000 + epoch)
    return DataLoader(dataset, batch_size=CONFIG['train']['batch_size'],
                      shuffle=shuffle, drop_last=drop_last,
                      generator=generator, num_workers=0, pin_memory=True)


def sha_array(values):
    values = np.ascontiguousarray(values)
    digest = hashlib.sha256()
    digest.update(str(values.shape).encode('ascii'))
    digest.update(str(values.dtype).encode('ascii'))
    digest.update(values.tobytes())
    return digest.hexdigest()


DATA_FINGERPRINT = {
    'clean_512': sha_array(TRAIN_CLEAN),
    'train_indices': sha_array(TRAIN_INDICES),
    'validation_indices': sha_array(VAL_INDICES),
    'mask_ids': sha_array(MIX_MASK_IDS),
    'noise_levels': sha_array(MIX_LEVELS),
    'bw_noise': sha_array(NOISE['BW']),
    'em_noise': sha_array(NOISE['EM']),
    'ma_noise': sha_array(NOISE['MA']),
}
(OUT / 'data_fingerprint.json').write_text(
    json.dumps(DATA_FINGERPRINT, indent=2), encoding='utf-8')
manifest = TRAIN_META[['sample_id', 'record_id']].copy()
manifest['split'] = np.where(
    np.isin(np.arange(len(TRAIN_CLEAN)), TRAIN_INDICES), 'train', 'validation')
manifest['mask'] = np.asarray(MIX_NAMES, dtype=object)[MIX_MASK_IDS]
manifest['noise_level_r'] = MIX_LEVELS
manifest.to_csv(OUT / 'mixed_pair_manifest.csv', index=False)

probe_clean, probe_noisy = next(iter(make_loader(
    'MIX7', PROTOCOLS_TO_TRAIN[0], TRAIN_INDICES[:128], False, False)))
assert probe_clean.shape[1:] == (1, LENGTH) and probe_noisy.shape == probe_clean.shape
assert torch.isfinite(probe_clean).all() and torch.isfinite(probe_noisy).all()
print('MIX7 loader probe:', tuple(probe_clean.shape), tuple(probe_noisy.shape))


## 10. Train/resume sáu experiment

In [ ]:
def experiment_stem(artifact, protocol):
    return f'u2_{artifact.lower()}_protocol{protocol}_{CHECKPOINT_TAG}'


def checkpoint_candidates(artifact, protocol):
    filename = f'last_{experiment_stem(artifact, protocol)}.pth'
    candidates = [TRAIN_DIR / filename]
    root = Path('/kaggle/input')
    if root.exists():
        candidates.extend(root.rglob(filename))
    return [path for path in candidates if path.is_file()]


def copy_prior_sidecars(checkpoint_path, artifact, protocol):
    stem = experiment_stem(artifact, protocol)
    for filename in (f'best_{stem}.pth', f'training_{stem}.csv'):
        source = checkpoint_path.parent / filename
        target = TRAIN_DIR / filename
        if source.is_file() and source.resolve() != target.resolve():
            shutil.copy2(source, target)


def save_state(model, optimizer, scheduler, ema, artifact, protocol, epoch, best_val, filename):
    ema_model_state = {key: value.detach().cpu().clone()
                       for key, value in model.state_dict().items()}
    for key, value in ema.state_dict().items():
        ema_model_state[key] = value.detach().cpu().clone()
    payload = {
        'state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'ema_state_dict': ema.state_dict(),
        'ema_model_state_dict': ema_model_state,
        'data_fingerprint': DATA_FINGERPRINT,
        'ablation_id': 'U2',
        'artifact': artifact,
        'protocol': protocol,
        'epoch': epoch,
        'best_validation_loss': best_val,
        'parameter_count': sum(p.numel() for p in model.parameters()),
        'config': CONFIG,
        'train_records': TRAIN_RECORDS,
        'test_records': TEST_RECORDS,
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state': torch.cuda.get_rng_state(),
    }
    target = TRAIN_DIR / filename
    temporary = target.with_suffix(target.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(target)


def validation_loss(model, loader, artifact, protocol):
    cpu_state = torch.get_rng_state()
    cuda_state = torch.cuda.get_rng_state()
    artifact_seed = {'MIX7': 44}[artifact]
    torch.manual_seed(SEED + artifact_seed * 10_000 + protocol)
    torch.cuda.manual_seed_all(SEED + artifact_seed * 10_000 + protocol)
    model.eval()
    sums = {'total': 0.0, 'time': 0.0, 'frequency': 0.0,
            'frequency_unweighted': 0.0, 'frequency_weight': 0.0}
    examples = 0
    with torch.no_grad():
        for clean, noisy in loader:
            clean, noisy = clean.to(DEVICE), noisy.to(DEVICE)
            components = model.loss_components(clean, noisy)
            for name in sums:
                sums[name] += float(components[name]) * len(clean)
            examples += len(clean)
    torch.set_rng_state(cpu_state)
    torch.cuda.set_rng_state(cuda_state)
    return {name: value / examples for name, value in sums.items()}


def train_experiment(artifact, protocol):
    stem = experiment_stem(artifact, protocol)
    last_name, best_name = f'last_{stem}.pth', f'best_{stem}.pth'
    log_path = TRAIN_DIR / f'training_{stem}.csv'
    model = ResearchLossDDPM(UNet1D(2, 80, 128, 1), CONFIG, DEVICE).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['train']['lr'])
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=150, gamma=0.1)
    ema = EMA(0.9)
    ema.register(model)
    val_loader = make_loader(artifact, protocol, VAL_INDICES, False, True)
    candidates = checkpoint_candidates(artifact, protocol)
    if candidates:
        payloads = [(torch.load(path, map_location='cpu', weights_only=False), path)
                    for path in candidates]
        payload, checkpoint_path = max(payloads, key=lambda pair: int(pair[0]['epoch']))
        if payload.get('ablation_id') != 'U2' or payload.get('artifact') != artifact or int(payload.get('protocol', -1)) != protocol:
            raise RuntimeError(f'Checkpoint identity mismatch: {checkpoint_path}')
        if (payload.get('config') != CONFIG or
                tuple(payload.get('train_records', ())) != TRAIN_RECORDS or
                payload.get('data_fingerprint') != DATA_FINGERPRINT):
            raise RuntimeError(f'Checkpoint configuration/data mismatch: {checkpoint_path}')
        model.load_state_dict(payload['state_dict'], strict=True)
        optimizer.load_state_dict(payload['optimizer_state_dict'])
        scheduler.load_state_dict(payload['scheduler_state_dict'])
        ema.load_state_dict({key: value.to(DEVICE)
                             for key, value in payload['ema_state_dict'].items()})
        if 'torch_rng_state' in payload:
            torch.set_rng_state(payload['torch_rng_state'])
        if payload.get('cuda_rng_state') is not None:
            torch.cuda.set_rng_state(payload['cuda_rng_state'])
        start_epoch = int(payload['epoch']) + 1
        best_val = float(payload['best_validation_loss'])
        copy_prior_sidecars(checkpoint_path, artifact, protocol)
        print(f'{artifact}/P{protocol}: resume epoch {start_epoch + 1} from {checkpoint_path}')
    else:
        start_epoch, best_val = 0, float('inf')
        print(f'{artifact}/P{protocol}: new run')

    stop_epoch = min(TARGET_EPOCHS, start_epoch + EPOCHS_PER_EXPERIMENT_THIS_SESSION)
    for epoch in range(start_epoch, stop_epoch):
        tick = time.perf_counter()
        loader = make_loader(artifact, protocol, TRAIN_INDICES, True, True, epoch)
        model.train()
        sums = {'total': 0.0, 'time': 0.0, 'frequency': 0.0}
        examples = 0
        for clean, noisy in tqdm(loader, desc=f'{artifact}/P{protocol} epoch {epoch + 1}', leave=False):
            clean, noisy = clean.to(DEVICE), noisy.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            components = model.loss_components(clean, noisy)
            if not torch.isfinite(components['total']):
                raise FloatingPointError(f'Nonfinite loss: {artifact}/P{protocol}/epoch{epoch + 1}')
            components['total'].backward()
            gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if not torch.isfinite(gradient_norm):
                raise FloatingPointError(f'Nonfinite gradient: {artifact}/P{protocol}/epoch{epoch + 1}')
            optimizer.step()
            ema.update(model)
            for name in sums:
                sums[name] += float(components[name].detach()) * len(clean)
            examples += len(clean)
        scheduler.step()
        train_values = {name: value / examples for name, value in sums.items()}
        val_values = validation_loss(model, val_loader, artifact, protocol)
        val = val_values['total']
        if not all(np.isfinite(value) for value in val_values.values()):
            raise FloatingPointError(f'Nonfinite validation: {artifact}/P{protocol}/epoch{epoch + 1}')
        improved = val < best_val
        if improved:
            best_val = val
        row = {
            'artifact': artifact, 'protocol': protocol, 'epoch': epoch + 1,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'train_loss': train_values['total'],
            'train_time_loss': train_values['time'],
            'train_frequency_loss': train_values['frequency'],
            'validation_loss': val,
            'validation_time_loss': val_values['time'],
            'validation_frequency_loss': val_values['frequency'],
            'validation_frequency_unweighted': val_values['frequency_unweighted'],
            'validation_frequency_weight': val_values['frequency_weight'],
            'best_validation_loss': best_val,
            'seconds': time.perf_counter() - tick,
        }
        pd.DataFrame([row]).to_csv(log_path, mode='a', header=not log_path.exists(), index=False)
        save_state(model, optimizer, scheduler, ema, artifact, protocol, epoch, best_val, last_name)
        if improved:
            save_state(model, optimizer, scheduler, ema, artifact, protocol, epoch, best_val, best_name)
        print(row)
        torch.cuda.empty_cache()
        gc.collect()
    return {'artifact': artifact, 'protocol': protocol,
            'completed_epochs': stop_epoch, 'target_epochs': TARGET_EPOCHS,
            'best_validation_loss': best_val}


RUN_STATUS = []
for artifact in ARTIFACTS_TO_TRAIN:
    for protocol in PROTOCOLS_TO_TRAIN:
        RUN_STATUS.append(train_experiment(artifact, protocol))
status = pd.DataFrame(RUN_STATUS)
status.to_csv(OUT / 'session_status.csv', index=False)
display(status)


## 11. Đồ thị và đóng gói output

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for log_path in sorted(TRAIN_DIR.glob('training_*.csv')):
    frame = pd.read_csv(log_path).drop_duplicates('epoch', keep='last').sort_values('epoch')
    label = f"{frame.artifact.iloc[0]}/P{int(frame.protocol.iloc[0])}"
    axes[0].plot(frame.epoch, frame.train_loss, label=f'{label} train')
    axes[0].plot(frame.epoch, frame.validation_loss, '--', label=f'{label} val')
    axes[1].plot(frame.epoch, frame.best_validation_loss, label=label)
axes[0].set(title='Training and validation loss', xlabel='Epoch', ylabel='Total loss')
axes[1].set(title='Best validation loss', xlabel='Epoch', ylabel='Loss')
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(OUT / 'training_curves.png', dpi=160)
plt.show()

checkpoint_rows = []
for path in sorted(TRAIN_DIR.glob('*.pth')):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    checkpoint_rows.append({
        'file': path.name, 'ablation_id': payload.get('ablation_id'),
        'parameters': payload.get('parameter_count'), 'artifact': payload.get('artifact'),
        'protocol': payload.get('protocol'), 'epoch': int(payload.get('epoch', -1)) + 1,
        'best_validation_loss': payload.get('best_validation_loss'),
        'sha256': sha256(path), 'size_mb': path.stat().st_size / 1024**2,
    })
checkpoint_manifest = pd.DataFrame(checkpoint_rows)
checkpoint_manifest.to_csv(OUT / 'checkpoint_manifest.csv', index=False)
display(checkpoint_manifest)

archive_path = Path('/kaggle/working/qtdb_u2_all_noise_512_100ep_outputs.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(OUT))
print('Output:', OUT, '| archive:', archive_path)


## Checkpoint and comparison contract

Best checkpoint: `best_u2_mix7_protocol1_mixed512_100ep_v1.pth`.
It stores model and EMA states, U2 identity, training config,
epoch, parameter count and the data fingerprint. Every U0–U4 notebook must
produce the **same** `data_fingerprint.json`; compare those files before
claiming a paired architecture ablation. Use the EMA state and one common
DDIM/test/metric pipeline. Existing specialist BW/EM/MA checkpoints and the
3600-sample benchmark checkpoint are not members of this MIX7 ablation.
